In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np
import requests
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M5, 0, 200)
#     rates_frame = pd.DataFrame(rates)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
    rates_frame['ema1'] =rates_frame['close'].ewm(span=9, adjust=False).mean()
    rates_frame['ema2'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
    rates_frame['ema3'] =rates_frame['close'].ewm(span=50, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
#     rates_frame['rsi'] = get_rsi(rates_frame['close'], 21)
# #     rates_frame['rsi'] = get_rsi(rates_frame['close'], r)
#     rates_frame['sma'] = rates_frame['close'].rolling(window=50).mean()

#     rates_frame['sma1'] = rates_frame['close'].rolling(window=9).mean()
#     rates_frame['sma2'] = rates_frame['close'].rolling(window=21).mean()
#     rates_frame = rates_frame[rates_frame['sma1'].notna()]
#     rates_frame = rates_frame[rates_frame['sma2'].notna()]

    # Calculate Supertren
    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0
    old = 0
    old_pp = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            if a.iloc[-2].close < a.iloc[-2].ema1 and a.iloc[-2].close < a.iloc[-2].ema2 and \
                a.iloc[-3].close < a.iloc[-3].ema1 and a.iloc[-3].close < a.iloc[-3].ema2 and direction(a, -3) == 0 and \
                (a.iloc[-4].close > a.iloc[-4].ema1 or a.iloc[-4].open > a.iloc[-4].ema1 or \
                a.iloc[-4].close > a.iloc[-4].ema2 or a.iloc[-4].open > a.iloc[-4].ema2):
                if a.iloc[-5].ema1 > a.iloc[-5].ema2:
                    if a.iloc[-5].close > a.iloc[-5].ema1 or a.iloc[-5].open > a.iloc[-5].ema1:
                        message = f"Sell BTC --> {a.iloc[-1].name}"
                        print(a.iloc[-2])
                        print(f"close -- {a.iloc[-2].close} ## ema1--{a.iloc[-2].ema1} ## ema2--{a.iloc[-2].ema2} ## {a.iloc[-2].name}")
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close
                        sell_check = 0
                else:
                    if a.iloc[-5].close > a.iloc[-5].ema2 or a.iloc[-5].open > a.iloc[-5].ema2:
                        message = f"Sell BTC --> {a.iloc[-1].name}"
                        print(a.iloc[-2])
                        print(f"close -- {a.iloc[-2].close} ## ema1--{a.iloc[-2].ema1} ## ema2--{a.iloc[-2].ema2} ## {a.iloc[-2].name}")
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close
                        sell_check = 0
                
            elif a.iloc[-2].close > a.iloc[-2].ema1 and a.iloc[-2].close > a.iloc[-2].ema2 and \
                a.iloc[-3].close > a.iloc[-3].ema1 and a.iloc[-3].close > a.iloc[-3].ema2 and direction(a, -3) == 1 and \
                (a.iloc[-4].close < a.iloc[-4].ema1 or a.iloc[-4].open < a.iloc[-4].ema1 or \
                a.iloc[-4].close < a.iloc[-4].ema2 or a.iloc[-4].open < a.iloc[-4].ema2):
                if a.iloc[-5].ema1 < a.iloc[-5].ema2:
                    if a.iloc[-5].close < a.iloc[-5].ema1 or a.iloc[-5].open < a.iloc[-5].ema1:
                        message = f"BUY BTC --> {a.iloc[-1].name}"
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close
                else:
                    if a.iloc[-5].close < a.iloc[-5].ema2 or a.iloc[-5].open < a.iloc[-5].ema2:
                        message = f"BUY BTC --> {a.iloc[-1].name}"
                        url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                        r = requests.get(url)
                        print(r.json())
                        old = a.iloc[-2].close

for symbol in ['BTCUSD']:
    run(symbol)
    

BTCUSD
open     57725.630000
high     57725.630000
low      57554.860000
close    57583.720000
ema1     57757.315962
ema2     57716.021866
ema3     57261.583005
Name: 2024-09-11 21:45:00, dtype: float64
close -- 57583.72 ## ema1--57757.31596182819 ## ema2--57716.02186590829 ## 2024-09-11 21:45:00
{'ok': True, 'result': {'message_id': 220, 'from': {'id': 7227666723, 'is_bot': True, 'first_name': 'signal', 'username': 'usdema_bot'}, 'chat': {'id': 220684438, 'first_name': 'Animesh', 'last_name': 'Verma', 'username': 'xicor', 'type': 'private'}, 'date': 1726080600, 'text': 'Sell BTC --> 2024-09-11 21:50:00'}}
